# E791 $D^+\to\pi^-\pi^+\pi^+$ — coefficient closure with uniform Dalitz MC normalization

This is the Monte Carlo counterpart of notebook 02. The physics model and fit setup are unchanged. The normalization sample is drawn uniformly in physical Dalitz area with `DalitzMC`.

Before fitting, this notebook compares the MC normalization matrix directly with a high-resolution `DalitzGrid` reference. When comparing likelihoods, each method is referenced to its own truth value, so only $\Delta$NLL is compared and any method-dependent additive normalization constant cancels.


In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from dalitzplotfitter import (
    DalitzMC, DalitzGrid, DecayChannel, DecayModel, Minimizer, NonResonant,
    Parameter, RealImag, Resonance, enable_x64, weighted_resample,
)
enable_x64()


In [ ]:
channel=DecayChannel("D+",("pi-","pi+","pi+"))
fit2_polar={
    "sigma":(1.17,205.7),"rho770":(1.00,0.0),"NR":(0.48,57.3),
    "f0_980":(0.43,165.0),"f2_1270":(0.76,57.3),
    "f0_1370":(0.26,105.4),"rho1450":(0.14,319.1),
}
def polar_to_xy(r,p):
    p=np.deg2rad(p); return r*np.cos(p),r*np.sin(p)
def internal_xy(name):
    r,p=fit2_polar[name]
    if name=="NR": p+=180.0
    return polar_to_xy(r,p)
truth_xy={n:internal_xy(n) for n in fit2_polar}
truth={}
def free_coeff(name):
    x,y=truth_xy[name]; truth[f"{name}.x"]=float(x); truth[f"{name}.y"]=float(y)
    return RealImag(Parameter.coefficient(f"{name}.x",0.0,owner=name,step=0.01),Parameter.coefficient(f"{name}.y",0.0,owner=name,step=0.01))
c={"sigma":free_coeff("sigma"),"rho770":RealImag(1.0,0.0),"NR":free_coeff("NR"),"f0_980":free_coeff("f0_980"),"f2_1270":free_coeff("f2_1270"),"f0_1370":free_coeff("f0_1370"),"rho1450":free_coeff("rho1450")}
model=DecayModel(channel,[
    Resonance("sigma",(0,1),c["sigma"],mass=0.4780,width=0.3240,spin=0,resonance_radius=3.0,parent_radius=3.0),
    Resonance("rho770",(0,1),c["rho770"],mass=0.7693,width=0.1502,spin=1,resonance_radius=3.0,parent_radius=3.0),
    Resonance("f0_980",(0,1),c["f0_980"],mass=0.9750,width=0.0440,spin=0,resonance_radius=3.0,parent_radius=3.0),
    Resonance("f2_1270",(0,1),c["f2_1270"],mass=1.2750,width=0.1850,spin=2,resonance_radius=3.0,parent_radius=3.0),
    Resonance("f0_1370",(0,1),c["f0_1370"],mass=1.4340,width=0.1730,spin=0,resonance_radius=3.0,parent_radius=3.0),
    Resonance("rho1450",(0,1),c["rho1450"],mass=1.4650,width=0.3100,spin=1,resonance_radius=3.0,parent_radius=3.0),
    NonResonant(c["NR"]),
])


## 1. Uniform Dalitz MC normalization


In [ ]:
N_NORM=1_000_000
norm=DalitzMC(channel.parent_mass,channel.daughter_masses).generate(N_NORM,seed=2027)
print("normalization points =",norm.size)
print("constant weights =",bool(jnp.all(norm.weights==norm.weights[0])))
print("weight =",float(norm.weights[0]))


## 2. Generate the same toy configuration as notebook 02


In [ ]:
N_POOL=1_000_000; N_DATA=100_000
pool=model.generate_phase_space(N_POOL,seed=2000)
truth_cache=model.prepare_cache(pool,norm)
truth_intensity,truth_norm=truth_cache.evaluate(truth)
data=weighted_resample(jax.random.key(791),pool,pool.weights*truth_intensity,N_DATA,replace=True)
cache=model.prepare_cache(data,norm)
def nll(values):
    intensity,normalization=cache.evaluate(values)
    return -jnp.sum(jnp.log(jnp.clip(intensity,min=1e-300)))+data.size*jnp.log(normalization)


## 3. MC versus deterministic-grid normalization matrix

For a coefficient-only fit the entire normalization dependence is contained in the fixed Hermitian interference matrix. If the MC implementation is correct, the normalized MC matrix should agree with the grid matrix within ordinary Monte Carlo fluctuations.


In [ ]:
GRID_N=1000
grid_norm=DalitzGrid(channel.parent_mass,channel.daughter_masses,resolution=GRID_N).sample()
grid_cache=model.prepare_cache(data,grid_norm)

M_mc=np.asarray(cache.normalization_matrix_fixed)
M_grid=np.asarray(grid_cache.normalization_matrix_fixed)
delta=M_mc-M_grid
mask=np.abs(M_grid)>1e-6

print("max |M_MC-M_grid| =",float(np.max(np.abs(delta))))
print("RMS |M_MC-M_grid| =",float(np.sqrt(np.mean(np.abs(delta)**2))))
print("max relative difference (|M_grid|>1e-6) =",float(np.max(np.abs(delta[mask])/np.abs(M_grid[mask]))))
print("MC Hermitian residual =",float(np.max(np.abs(M_mc-M_mc.conj().T))))
print("grid Hermitian residual =",float(np.max(np.abs(M_grid-M_grid.conj().T))))
print("MC eigenvalues =",np.linalg.eigvalsh(M_mc))
print("grid eigenvalues =",np.linalg.eigvalsh(M_grid))


## 4. Compare truth-referenced $\Delta$NLL

Do not compare the absolute MC and grid NLL values directly. Define, separately for each normalization method,

$$\Delta\mathrm{NLL}(\theta)=\mathrm{NLL}(\theta)-\mathrm{NLL}(\theta_{truth}).$$

This removes any additive constant associated with the normalization convention.


In [ ]:
def nll_grid(values):
    intensity,normalization=grid_cache.evaluate(values)
    return -jnp.sum(jnp.log(jnp.clip(intensity,min=1e-300)))+data.size*jnp.log(normalization)

nll_mc_truth=float(nll(truth))
nll_grid_truth=float(nll_grid(truth))

def delta_nll_mc(values):
    return float(nll(values))-nll_mc_truth

def delta_nll_grid(values):
    return float(nll_grid(values))-nll_grid_truth

rng=np.random.default_rng(314159)
start_values={p.name:float(rng.uniform(-2.5,2.5)) for p in model.parameters if not p.fixed}

print("DeltaNLL_MC(truth)   =",delta_nll_mc(truth))
print("DeltaNLL_grid(truth) =",delta_nll_grid(truth))
print("DeltaNLL_MC(start)   =",delta_nll_mc(start_values))
print("DeltaNLL_grid(start) =",delta_nll_grid(start_values))


## 5. Compare the truth-referenced NLL surfaces from start to truth

This is not a minimization. Both curves are referenced to their own truth NLL, so their vertical offsets are physically irrelevant and removed.


In [ ]:
free_names=[p.name for p in model.parameters if not p.fixed]
ts=np.linspace(0.0,1.0,81)
mc_scan=[]; grid_scan=[]
for t in ts:
    point={name:(1-t)*start_values[name]+t*truth[name] for name in free_names}
    mc_scan.append(delta_nll_mc(point))
    grid_scan.append(delta_nll_grid(point))
mc_scan=np.asarray(mc_scan); grid_scan=np.asarray(grid_scan)
fig,ax=plt.subplots(figsize=(8,5))
ax.plot(ts,mc_scan,label="uniform MC")
ax.plot(ts,grid_scan,linestyle="--",label="grid")
ax.axhline(0.0,linewidth=1.0)
ax.set_xlabel("t: start -> truth"); ax.set_ylabel(r"$\Delta$NLL relative to truth"); ax.legend(); plt.show()


## 6. Gradient check and one MC-normalized fit


In [ ]:
minimizer=Minimizer(nll,model.parameters,tolerance=1e-4,verbose=2)
gradient_check=minimizer.check_gradient(start_values,step_scale=1e-5,print_table=True)


In [ ]:
result=minimizer.fit(start_values=start_values,simplex=False,ncall=100000)
fit_values={p.name:float(result.values[p.name]) for p in model.parameters if not p.fixed}
print("valid=",bool(result.valid))
print("DeltaNLL_MC(fit) =",delta_nll_mc(fit_values))
print("EDM=",float(result.fmin.edm))
print(f"{'parameter':16s} {'truth':>10s} {'fit':>10s} {'error':>10s} {'pull':>9s}")
for p in model.parameters:
    if p.fixed: continue
    t=float(truth[p.name]); f=float(result.values[p.name]); e=float(result.errors[p.name]); pull=(f-t)/e
    print(f"{p.name:16s} {t:10.5f} {f:10.5f} {e:10.5f} {pull:9.3f}")


## 7. Cross-check the fitted point with truth-referenced $\Delta$NLL

The same parameter point is evaluated in both objectives, but each objective is referenced to its own truth value. A good closure solution should have $\Delta$NLL near or below zero in its own likelihood.


In [ ]:
print("DeltaNLL_MC(MC fit)   =",delta_nll_mc(fit_values))
print("DeltaNLL_grid(MC fit) =",delta_nll_grid(fit_values))
